<a href="https://colab.research.google.com/github/alpacaYiChun/ML/blob/master/MovieLens_GCN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# KG-GCN on MovieLens (movies+genres+tags) with:
# 1) cached allow_list per user
# 2) HARD negative mining: genre filter -> tag-embedding ANN prefilter -> model-score topK
# 3) multi-neg BPR + L2 reg
# PLUS stability:
# - scoring uses cosine + temperature + clamp
# - gradient clipping (+ optional AMP)
# - hard mining strength schedules up over first 10 epochs

import os, re, random, zipfile, urllib.request
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# ---- NLP embeddings for tags ----
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

# ----------------------------
# Repro & device
# ----------------------------
seed = 42
random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# ----------------------------
# Download / load MovieLens
# ----------------------------
DATA_DIR = "./data"
ML_NAME  = "ml-latest-small"
ML_DIR   = os.path.join(DATA_DIR, ML_NAME)
ZIP_PATH = os.path.join(DATA_DIR, f"{ML_NAME}.zip")
URL      = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.exists(ML_DIR):
    if not os.path.exists(ZIP_PATH):
        print("Downloading MovieLens...")
        urllib.request.urlretrieve(URL, ZIP_PATH)
    print("Extracting...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR)

ratings = pd.read_csv(os.path.join(ML_DIR, "ratings.csv"))
movies  = pd.read_csv(os.path.join(ML_DIR, "movies.csv"))
tags_df = pd.read_csv(os.path.join(ML_DIR, "tags.csv")) if os.path.exists(os.path.join(ML_DIR, "tags.csv")) else None

# implicit feedback: rating >= 4.0
ratings = ratings[ratings["rating"] >= 4.0].copy()
ratings.sort_values(["userId", "timestamp"], inplace=True)

# id -> contiguous (only users/movies in filtered ratings)
user_ids  = sorted(ratings["userId"].unique().tolist())
movie_ids = sorted(ratings["movieId"].unique().tolist())
uid2u = {uid:i for i, uid in enumerate(user_ids)}
mid2m = {mid:i for i, mid in enumerate(movie_ids)}
ratings["u"] = ratings["userId"].map(uid2u)
ratings["m"] = ratings["movieId"].map(mid2m)

num_users  = len(user_ids)
num_movies = len(movie_ids)
print("num_users:", num_users, "num_movies:", num_movies, "num_pos_interactions:", len(ratings))

# ----------------------------
# Split: leave-one-out per user (test=latest pos, val=second-latest pos if exists)
# ----------------------------
user_hist = defaultdict(list)
for u, m, ts in ratings[["u","m","timestamp"]].itertuples(index=False):
    user_hist[int(u)].append((int(ts), int(m)))
for u in user_hist:
    user_hist[u].sort(key=lambda x: x[0])

train_pos = defaultdict(set)
val_pos   = {}
test_pos  = {}

for u, seq in user_hist.items():
    ms = [m for _, m in seq]
    if len(ms) >= 3:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
        for m in ms[:-2]:
            train_pos[u].add(m)
    elif len(ms) == 2:
        test_pos[u] = ms[-1]
        val_pos[u]  = ms[-2]
    elif len(ms) == 1:
        test_pos[u] = ms[-1]

eval_users = [u for u in range(num_users) if u in test_pos]
print("eval_users:", len(eval_users))

# train edges (train only)
train_edges = []
for u in range(num_users):
    for m in train_pos.get(u, []):
        train_edges.append((u, m))
print("train_edges:", len(train_edges))

# ----------------------------
# Build internal KG: movie-genres, movie-tags
# ----------------------------
movies_sub = movies[movies["movieId"].isin(movie_ids)].copy()
movies_sub["m"] = movies_sub["movieId"].map(mid2m)

# Genres
genre_set = set()
movie_genres = defaultdict(list)  # m -> [genre str]
for m, gstr in movies_sub[["m","genres"]].itertuples(index=False):
    m = int(m)
    if isinstance(gstr, str) and gstr != "(no genres listed)":
        for g in gstr.split("|"):
            genre_set.add(g)
            movie_genres[m].append(g)

genres = sorted(list(genre_set))
gid2g = {g:i for i,g in enumerate(genres)}
num_genres = len(genres)

# Tags
movie_tags = defaultdict(list)    # m -> [tag_norm str]
num_tags = 0
tag_list = []
tid2t = {}

if tags_df is not None and len(tags_df) > 0:
    tags_sub = tags_df[tags_df["movieId"].isin(movie_ids)].copy()

    def norm_tag(s):
        s = str(s).lower().strip()
        s = re.sub(r"\s+", " ", s)
        return s

    tags_sub["tag_norm"] = tags_sub["tag"].apply(norm_tag)
    tag_counts = tags_sub["tag_norm"].value_counts()
    TOP_TAGS = 3000
    kept_tags = set(tag_counts.head(TOP_TAGS).index.tolist())
    tags_sub = tags_sub[tags_sub["tag_norm"].isin(kept_tags)]

    tag_list = sorted(tags_sub["tag_norm"].unique().tolist())
    tid2t = {t:i for i,t in enumerate(tag_list)}
    num_tags = len(tag_list)

    # movie -> tags
    for mid, t in tags_sub[["movieId","tag_norm"]].itertuples(index=False):
        mid = int(mid)
        if mid in mid2m:
            m = int(mid2m[mid])
            movie_tags[m].append(t)
    for m in list(movie_tags.keys()):
        movie_tags[m] = sorted(list(set(movie_tags[m])))

print("num_genres:", num_genres, "num_tags:", num_tags)

# ----------------------------
# Build big graph: [users][movies][genres][tags]
# ----------------------------
OFF_U = 0
OFF_M = OFF_U + num_users
OFF_G = OFF_M + num_movies
OFF_T = OFF_G + num_genres
num_nodes = OFF_T + num_tags

edges_src, edges_dst = [], []
def add_undirected(a, b):
    edges_src.append(a); edges_dst.append(b)
    edges_src.append(b); edges_dst.append(a)

# user-movie edges (train only)
for u, m in train_edges:
    add_undirected(OFF_U + int(u), OFF_M + int(m))

# movie-genre edges
for m in range(num_movies):
    for g in movie_genres.get(m, []):
        add_undirected(OFF_M + m, OFF_G + gid2g[g])

# movie-tag edges
if num_tags > 0:
    for m in range(num_movies):
        for t in movie_tags.get(m, []):
            add_undirected(OFF_M + m, OFF_T + tid2t[t])

edge_index = torch.tensor([edges_src, edges_dst], dtype=torch.long, device=device)
deg = torch.bincount(edge_index[0], minlength=num_nodes).float()
deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)
norm = deg_inv_sqrt[edge_index[0]] * deg_inv_sqrt[edge_index[1]]  # [E]
print("num_nodes:", num_nodes, "num_edges(directed):", edge_index.size(1))

# ----------------------------
# Model: relation-agnostic LightGCN-style
# ----------------------------
class KGGcnRec(nn.Module):
    def __init__(self, num_nodes, num_users, num_movies, dim=128, layers=4, dropout=0.1):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, dim)
        nn.init.normal_(self.emb.weight, std=0.01)
        self.layers = layers
        self.dropout = dropout
        self.num_users = num_users
        self.num_movies = num_movies

    def propagate(self, x, edge_index, norm):
        src, dst = edge_index[0], edge_index[1]
        msg = x[src] * norm.unsqueeze(-1)
        out = torch.zeros_like(x)
        out.index_add_(0, dst, msg)
        return out

    def forward(self, edge_index, norm):
        x0 = self.emb.weight
        xs = [x0]
        x = x0
        for _ in range(self.layers):
            x = self.propagate(x, edge_index, norm)
            x = F.dropout(x, p=self.dropout, training=self.training)
            xs.append(x)
        x_final = torch.mean(torch.stack(xs, dim=0), dim=0)
        user_e  = x_final[OFF_U:OFF_U + num_users]
        movie_e = x_final[OFF_M:OFF_M + num_movies]
        return user_e, movie_e

# Cosine+temperature+clamp multi-neg BPR + L2 reg
def bpr_multi_loss_l2(u_e, pos_e, neg_e, l2_lambda=1e-4, tau=0.25, clamp=10.0):
    """
    u_e:   [B,D]
    pos_e: [B,D]
    neg_e: [B,N,D]
    """
    u = F.normalize(u_e, dim=-1)
    p = F.normalize(pos_e, dim=-1)
    n = F.normalize(neg_e, dim=-1)

    pos = (u * p).sum(dim=-1, keepdim=True) / tau      # [B,1]
    neg = (u.unsqueeze(1) * n).sum(dim=-1) / tau       # [B,N]

    diff = (pos - neg).clamp(min=-clamp, max=clamp)    # [B,N]
    bpr = -F.logsigmoid(diff).mean()

    l2 = (
        u_e.pow(2).sum(dim=-1) +
        pos_e.pow(2).sum(dim=-1) +
        neg_e.pow(2).sum(dim=-1).mean(dim=1)
    ).mean()
    return bpr + l2_lambda * l2

model = KGGcnRec(num_nodes, num_users, num_movies, dim=128, layers=4, dropout=0.1).to(device)
opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-6)

# ----------------------------
# Structures + allow_list cache
# ----------------------------
train_pos_set = {u: set(train_pos.get(u, set())) for u in range(num_users)}
train_pos_np  = {u: np.fromiter(train_pos_set[u], dtype=np.int64) for u in range(num_users)}

def blocked_set(u):
    b = set(train_pos_set[u])
    if u in val_pos:  b.add(val_pos[u])
    if u in test_pos: b.add(test_pos[u])
    return b

all_movies = np.arange(num_movies, dtype=np.int64)

allow_list = {}
for u in range(num_users):
    b = blocked_set(u)
    mask = np.ones(num_movies, dtype=bool)
    if len(b) > 0:
        mask[np.fromiter(b, dtype=np.int64)] = False
    allow_list[u] = all_movies[mask]

def sample_from_allow(u, k):
    cand = allow_list[u]
    if len(cand) == 0:
        raise RuntimeError(f"user {u} has no available negatives")
    replace = len(cand) < k
    return np.random.choice(cand, size=k, replace=replace).astype(np.int64)

train_users = [u for u in range(num_users) if len(train_pos_set[u]) > 0]
if len(train_users) == 0:
    raise RuntimeError("No users have training positives after split.")

# ----------------------------
# Precompute movie genres
# ----------------------------
movie_genre_sets = [set(movie_genres.get(m, [])) for m in range(num_movies)]

# ----------------------------
# NLP embeddings for tags
# ----------------------------
if num_tags > 0:
    print("Building tag NLP embeddings...")
    st_model = SentenceTransformer("all-MiniLM-L6-v2", device=str(device))
    tag_text_emb = st_model.encode(
        tag_list, batch_size=256, show_progress_bar=True,
        convert_to_numpy=True, normalize_embeddings=True
    )
    tag_text_emb = torch.tensor(tag_text_emb, device=device, dtype=torch.float32)  # [T,dt]
    movie_tag_emb = torch.zeros((num_movies, tag_text_emb.size(1)), device=device)

    for m in range(num_movies):
        ts = movie_tags.get(m, [])
        if len(ts) > 0:
            idx = torch.tensor([tid2t[t] for t in ts], device=device, dtype=torch.long)
            movie_tag_emb[m] = tag_text_emb[idx].mean(dim=0)
    movie_tag_emb = F.normalize(movie_tag_emb, dim=-1)
else:
    tag_text_emb = None
    movie_tag_emb = None
    print("No tags found; tag-embedding filter will be skipped.")

# ----------------------------
# Recall@K
# ----------------------------
@torch.no_grad()
def recall_at_ks(user_e, movie_e, ks=(5,10,20,50), users=None):
    model.eval()
    if users is None:
        users = eval_users
    movie_e_t = movie_e.t()
    recalls = {k: [] for k in ks}

    for u in users:
        if u not in test_pos:
            continue
        gt = test_pos[u]
        scores = (user_e[u:u+1] @ movie_e_t).squeeze(0)
        seen = set(train_pos_set[u])
        if u in val_pos: seen.add(val_pos[u])
        if len(seen) > 0:
            scores[torch.tensor(list(seen), device=device, dtype=torch.long)] = -1e9
        topk = torch.topk(scores, k=min(max(ks), num_movies), largest=True).indices.detach().cpu().numpy()
        for k in ks:
            recalls[k].append(1.0 if gt in topk[:k] else 0.0)

    return {k: float(np.mean(recalls[k])) if len(recalls[k]) else 0.0 for k in ks}

# ----------------------------
# Hard mining helpers
# ----------------------------
@torch.no_grad()
def build_user_profiles(train_users):
    user_genres = {}
    for u in train_users:
        gs = set()
        for m in train_pos_set[u]:
            gs |= movie_genre_sets[m]
        user_genres[u] = gs

    user_tag_emb = None
    if movie_tag_emb is not None:
        user_tag_emb = torch.zeros((num_users, movie_tag_emb.size(1)), device=device)
        for u in train_users:
            ms = list(train_pos_set[u])
            if len(ms) > 0:
                user_tag_emb[u] = movie_tag_emb[torch.tensor(ms, device=device, dtype=torch.long)].mean(dim=0)
        user_tag_emb = F.normalize(user_tag_emb, dim=-1)

    return user_genres, user_tag_emb

def genre_filter_allowed(u, user_genres, min_overlap=1):
    cand = allow_list[u]
    ug = user_genres.get(u, set())
    if not ug:
        return cand
    out = []
    for m in cand:
        if len(movie_genre_sets[int(m)] & ug) >= min_overlap:
            out.append(int(m))
    return np.array(out, dtype=np.int64) if len(out) else cand

@torch.no_grad()
def build_hard_pools(user_e_det, movie_e_det, train_users, user_genres, user_tag_emb,
                     GENRE_MIN_OVERLAP=1, TAG_TOPN=800, HARD_TOPK=300):
    pools = {}
    for u in train_users:
        cand = genre_filter_allowed(u, user_genres, min_overlap=GENRE_MIN_OVERLAP)
        if len(cand) == 0:
            pools[u] = np.empty((0,), dtype=np.int64); continue

        if user_tag_emb is not None and movie_tag_emb is not None:
            cand_t = torch.tensor(cand, device=device, dtype=torch.long)
            sim = (movie_tag_emb[cand_t] * user_tag_emb[u].unsqueeze(0)).sum(dim=-1)  # cosine
            topn = min(TAG_TOPN, cand_t.numel())
            cand = cand_t[torch.topk(sim, k=topn, largest=True).indices].detach().cpu().numpy().astype(np.int64)

        cand_t2 = torch.tensor(cand, device=device, dtype=torch.long)
        # model score uses cosine too (stable)
        scores = (F.normalize(user_e_det[u], dim=-1).unsqueeze(0) * F.normalize(movie_e_det[cand_t2], dim=-1)).sum(dim=-1)
        topk = min(HARD_TOPK, cand_t2.numel())
        pools[u] = cand_t2[torch.topk(scores, k=topk, largest=True).indices].detach().cpu().numpy().astype(np.int64)

    return pools

def sample_hard_from_pool(u, pools, k):
    cand = pools.get(u, None)
    if cand is None or len(cand) == 0:
        return sample_from_allow(u, k)
    replace = len(cand) < k
    return np.random.choice(cand, size=k, replace=replace).astype(np.int64)

# ----------------------------
# Training config
# ----------------------------
BATCH      = 1024
EPOCHS     = 200
EVAL_EVERY = 1

NEG_PER_POS = 24
HARD_FRAC   = 0.6
L2_LAMBDA   = 1e-4

HARD_NEG = int(np.ceil(NEG_PER_POS * HARD_FRAC))
RAND_NEG = NEG_PER_POS - HARD_NEG
assert HARD_NEG / NEG_PER_POS >= 0.5

# Hard mining "target" difficulty
GENRE_MIN_OVERLAP_TGT = 1
TAG_TOPN_TGT          = 800
HARD_TOPK_TGT         = 300

# Stability
TAU = 0.25          # temperature for BPR (cosine/tau)
CLAMP = 0.5        # clamp pos-neg
MAX_GRAD_NORM = 0.5 # gradient clipping

# optional AMP
use_amp = (device.type == "cuda")
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

steps = max(1, len(train_edges) // BATCH)

def ramp_to_target(epoch, ramp_epochs, start_ratio=0.2):
    """
    Linear ramp factor in [start_ratio, 1.0] over first ramp_epochs.
    """
    if ramp_epochs <= 1:
        return 1.0
    t = min(max((epoch - 1) / (ramp_epochs - 1), 0.0), 1.0)
    return start_ratio + (1.0 - start_ratio) * t

RAMP_EPOCHS = 10   # user requested

print("\nTraining (cosine+tau+clamp + grad clip + hard ramp first 10 epochs)...")
for epoch in range(1, EPOCHS + 1):
    model.train()

    # Build embeddings once per epoch
    with torch.cuda.amp.autocast(enabled=use_amp):
        user_e, movie_e = model(edge_index, norm)

    user_e_det  = user_e.detach()
    movie_e_det = movie_e.detach()

    # profiles
    user_genres, user_tag_emb = build_user_profiles(train_users)

    # hard strength ramp
    s = ramp_to_target(epoch, RAMP_EPOCHS, start_ratio=0.2)
    GENRE_MIN_OVERLAP = GENRE_MIN_OVERLAP_TGT  # overlap is discrete; keep it stable
    TAG_TOPN_eff  = max(50, int(TAG_TOPN_TGT  * s))
    HARD_TOPK_eff = max(50, int(HARD_TOPK_TGT * s))

    hard_pools = build_hard_pools(
        user_e_det, movie_e_det, train_users,
        user_genres=user_genres, user_tag_emb=user_tag_emb,
        GENRE_MIN_OVERLAP=GENRE_MIN_OVERLAP,
        TAG_TOPN=TAG_TOPN_eff,
        HARD_TOPK=HARD_TOPK_eff
    )

    total_loss = 0.0

    for _ in range(steps):
        us = np.random.choice(train_users, size=BATCH, replace=True)

        pos_ms = np.empty(BATCH, dtype=np.int64)
        neg_ms = np.empty((BATCH, NEG_PER_POS), dtype=np.int64)

        for i, u in enumerate(us):
            u = int(u)
            plist = train_pos_np[u]
            pos_ms[i] = int(plist[np.random.randint(0, len(plist))])

            if HARD_NEG > 0:
                neg_ms[i, :HARD_NEG] = sample_hard_from_pool(u, hard_pools, HARD_NEG)
            if RAND_NEG > 0:
                neg_ms[i, HARD_NEG:] = sample_from_allow(u, RAND_NEG)

        us_t  = torch.tensor(us, device=device, dtype=torch.long)
        pos_t = torch.tensor(pos_ms, device=device, dtype=torch.long)
        neg_t = torch.tensor(neg_ms, device=device, dtype=torch.long)  # [B,N]

        uvec = user_e[us_t]
        pvec = movie_e[pos_t]
        nvec = movie_e[neg_t]

        with torch.cuda.amp.autocast(enabled=use_amp):
            total_loss = total_loss + bpr_multi_loss_l2(
                uvec, pvec, nvec,
                l2_lambda=L2_LAMBDA,
                tau=TAU,
                clamp=CLAMP
            )

    total_loss = total_loss / steps

    opt.zero_grad(set_to_none=True)

    if use_amp:
        scaler.scale(total_loss).backward()
        scaler.unscale_(opt)
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        scaler.step(opt)
        scaler.update()
    else:
        total_loss.backward()
        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        opt.step()

    if epoch % EVAL_EVERY == 0:
        model.eval()
        with torch.no_grad():
            ue, me = model(edge_index, norm)
            rec = recall_at_ks(ue, me, ks=(5,10,20,50), users=eval_users)

        print(f"Epoch {epoch:02d} | loss={float(total_loss.detach().cpu()):.4f} | "
              f"grad_norm={float(grad_norm):.3f} | hardRamp={s:.2f} "
              f"(TAG_TOPN={TAG_TOPN_eff}, HARD_TOPK={HARD_TOPK_eff}) | "
              f"R@5={rec[5]:.4f} R@10={rec[10]:.4f} R@20={rec[20]:.4f} R@50={rec[50]:.4f}")

print("\nDone.")

device: cpu
Extracting...
num_users: 609 num_movies: 6298 num_pos_interactions: 48580
eval_users: 609
train_edges: 47363
num_genres: 19 num_tags: 1415
num_nodes: 8341 num_edges(directed): 130808
Building tag NLP embeddings...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

/tmp/ipython-input-3193954063.py:437: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/tmp/ipython-input-3193954063.py:457: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):



Training (cosine+tau+clamp + grad clip + hard ramp first 10 epochs)...


/tmp/ipython-input-3193954063.py:506: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Epoch 01 | loss=0.6928 | grad_norm=0.235 | hardRamp=0.20 (TAG_TOPN=160, HARD_TOPK=60) | R@5=0.0033 R@10=0.0049 R@20=0.0115 R@50=0.0312
Epoch 02 | loss=0.6088 | grad_norm=0.234 | hardRamp=0.29 (TAG_TOPN=231, HARD_TOPK=86) | R@5=0.0131 R@10=0.0213 R@20=0.0460 R@50=0.0772
Epoch 03 | loss=0.5577 | grad_norm=0.130 | hardRamp=0.38 (TAG_TOPN=302, HARD_TOPK=113) | R@5=0.0213 R@10=0.0394 R@20=0.0575 R@50=0.1100
Epoch 04 | loss=0.5354 | grad_norm=0.093 | hardRamp=0.47 (TAG_TOPN=373, HARD_TOPK=140) | R@5=0.0263 R@10=0.0394 R@20=0.0739 R@50=0.1248
Epoch 05 | loss=0.5254 | grad_norm=0.075 | hardRamp=0.56 (TAG_TOPN=444, HARD_TOPK=166) | R@5=0.0312 R@10=0.0443 R@20=0.0739 R@50=0.1363
Epoch 06 | loss=0.5181 | grad_norm=0.060 | hardRamp=0.64 (TAG_TOPN=515, HARD_TOPK=193) | R@5=0.0279 R@10=0.0460 R@20=0.0755 R@50=0.1478
Epoch 07 | loss=0.5122 | grad_norm=0.048 | hardRamp=0.73 (TAG_TOPN=586, HARD_TOPK=220) | R@5=0.0279 R@10=0.0411 R@20=0.0805 R@50=0.1642
Epoch 08 | loss=0.5108 | grad_norm=0.043 | hardRam